In [1]:
import os
import shutil
from datetime import date
from pathlib import Path
from dotenv import load_dotenv
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip


In [2]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, ShortType, StringType,
    TimestampType, DateType, DecimalType,
)

SCHEMAS = {
    "usuarios": StructType([
        StructField("id",            IntegerType(),  False),
        StructField("nome",          StringType(),   False),
        StructField("email",         StringType(),   False),
        StructField("data_cadastro", DateType(),     False),
        StructField("pais",          StringType(),   False),
        StructField("created_at",    TimestampType(),False),
    ]),
    "planos": StructType([
        StructField("id",           IntegerType(),    False),
        StructField("nome",         StringType(),     False),
        StructField("preco_mensal", DecimalType(8,2), False),
        StructField("created_at",   TimestampType(),  False),
    ]),
    "artistas": StructType([
        StructField("id",         IntegerType(),  False),
        StructField("nome",       StringType(),   False),
        StructField("pais",       StringType(),   False),
        StructField("created_at", TimestampType(),False),
    ]),
    "albuns": StructType([
        StructField("id",              IntegerType(),  False),
        StructField("artista_id",      IntegerType(),  False),
        StructField("titulo",          StringType(),   False),
        StructField("ano_lancamento",  ShortType(),    False),
        StructField("created_at",      TimestampType(),False),
    ]),
    "musicas": StructType([
        StructField("id",         IntegerType(),  False),
        StructField("album_id",   IntegerType(),  False),
        StructField("artista_id", IntegerType(),  False),
        StructField("genero_id",  IntegerType(),  False),
        StructField("titulo",     StringType(),   False),
        StructField("duracao_ms", IntegerType(),  False),
        StructField("created_at", TimestampType(),False),
    ]),
    "generos": StructType([
        StructField("id",         IntegerType(),  False),
        StructField("nome",       StringType(),   False),
        StructField("created_at", TimestampType(),False),
    ]),
    "playlists": StructType([
        StructField("id",         IntegerType(),  False),
        StructField("usuario_id", IntegerType(),  False),
        StructField("nome",       StringType(),   False),
        StructField("created_at", TimestampType(),False),
    ]),
    "dispositivos": StructType([
        StructField("id",         IntegerType(),  False),
        StructField("usuario_id", IntegerType(),  False),
        StructField("tipo",       StringType(),   False),
        StructField("so",         StringType(),   False),
        StructField("created_at", TimestampType(),False),
    ]),
}


In [3]:
import os
import shutil
from datetime import date
from pathlib import Path
from dotenv import load_dotenv
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

load_dotenv()

is_docker = os.path.exists("/.dockerenv")

MINIO_HOST = "minio:9000" if is_docker else "localhost:9000"
MINIO_USER = os.getenv("MINIO_ROOT_USER",     "minioadmin")
MINIO_PASS = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

builder = (
    SparkSession.builder
    .appName("bronze-dimensoes")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

cliente_minio = Minio(
    MINIO_HOST,
    access_key=MINIO_USER,
    secret_key=MINIO_PASS,
    secure=False,
)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/21 16:59:45 WARN Utils: Your hostname, DESKTOP-S3JB25M, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/21 16:59:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/mnt/c/Users/felip/www/university/eng-dados/projeto-ed-final/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/felipe/.ivy2.5.2/cache
The jars for the packages stored in: /home/felipe/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a8de5ede-fe2e-4317-97ff-843a314a15f2;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf

In [4]:
buckets = {b.name for b in cliente_minio.list_buckets()}
print("Buckets disponíveis:", buckets)

if "bronze" not in buckets:
    cliente_minio.make_bucket("bronze")
    print("Bucket 'bronze' criado.")


Buckets disponíveis: {'silver', 'landing', 'gold', 'bronze'}


In [5]:
def download_landing_csv(tabela: str, cliente: Minio, destino_dir: Path) -> Path:
    """Baixa o CSV da Landing (MinIO) para um diretório local."""
    destino_dir.mkdir(parents=True, exist_ok=True)
    destino_arquivo = destino_dir / f"{tabela}.csv"
    cliente.fget_object("landing", f"{tabela}/{tabela}.csv", str(destino_arquivo))
    return destino_arquivo


def upload_delta_para_minio(local_dir: Path, bucket: str, prefix: str,
                            cliente: Minio) -> None:
    """Sobe recursivamente a estrutura Delta (parquet + _delta_log) para o MinIO."""
    for arq in sorted(local_dir.rglob("*")):
        if arq.is_file():
            object_name = f"{prefix}/{arq.relative_to(local_dir)}"
            cliente.fput_object(bucket, object_name, str(arq))
            print(f"  → {object_name}")


In [6]:
INGESTAO_DATE = str(date.today())
print(f"ingestao_date = {INGESTAO_DATE}")


ingestao_date = 2026-06-21


In [7]:
for tabela, schema in SCHEMAS.items():
    print(f"\n{'=' * 55}")
    print(f"Tabela: {tabela}")
    print(f"{'=' * 55}")

    local_csv_dir = Path(f"/tmp/landing_local/{tabela}")
    csv_path = download_landing_csv(tabela, cliente_minio, local_csv_dir)
    print(f"CSV baixado: {csv_path}")

    df = (
        spark.read
        .option("header", "true")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .option("dateFormat", "yyyy-MM-dd")
        .schema(schema)
        .csv(str(csv_path))
    )

    total = df.count()
    print(f"Registros lidos: {total:,}")

    df = df.withColumn("ingestao_date", F.lit(INGESTAO_DATE).cast("date"))

    local_delta_dir = Path(f"/tmp/bronze/{tabela}")
    if local_delta_dir.exists():
        shutil.rmtree(local_delta_dir)

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("ingestao_date")
        .save(str(local_delta_dir))
    )
    print(f"Delta escrito localmente em {local_delta_dir}")

    print(f"Enviando para MinIO bronze/dimensoes/{tabela}/")
    upload_delta_para_minio(
        local_delta_dir, "bronze", f"dimensoes/{tabela}", cliente_minio
    )
    print(f"{tabela} concluído — {total:,} registros na Bronze.")

print("\nPromoção Landing → Bronze concluída para todas as dimensões.")



Tabela: usuarios
CSV baixado: /tmp/landing_local/usuarios/usuarios.csv


Registros lidos: 3,000


26/06/21 17:00:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Delta escrito localmente em /tmp/bronze/usuarios
Enviando para MinIO bronze/dimensoes/usuarios/
  → dimensoes/usuarios/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/usuarios/_delta_log/.00000000000000000000.json.crc
  → dimensoes/usuarios/_delta_log/00000000000000000000.crc
  → dimensoes/usuarios/_delta_log/00000000000000000000.json
  → dimensoes/usuarios/ingestao_date=2026-06-21/.part-00000-d1d7bd60-7b4f-4e14-a886-996dc5fef9fc.c000.snappy.parquet.crc
  → dimensoes/usuarios/ingestao_date=2026-06-21/part-00000-d1d7bd60-7b4f-4e14-a886-996dc5fef9fc.c000.snappy.parquet
usuarios concluído — 3,000 registros na Bronze.

Tabela: planos
CSV baixado: /tmp/landing_local/planos/planos.csv
Registros lidos: 3


Delta escrito localmente em /tmp/bronze/planos
Enviando para MinIO bronze/dimensoes/planos/
  → dimensoes/planos/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/planos/_delta_log/.00000000000000000000.json.crc
  → dimensoes/planos/_delta_log/00000000000000000000.crc
  → dimensoes/planos/_delta_log/00000000000000000000.json
  → dimensoes/planos/ingestao_date=2026-06-21/.part-00000-207bb5e8-1a99-45f6-a997-e26da838b23b.c000.snappy.parquet.crc
  → dimensoes/planos/ingestao_date=2026-06-21/part-00000-207bb5e8-1a99-45f6-a997-e26da838b23b.c000.snappy.parquet
planos concluído — 3 registros na Bronze.

Tabela: artistas
CSV baixado: /tmp/landing_local/artistas/artistas.csv
Registros lidos: 500


...

Delta escrito localmente em /tmp/bronze/albuns
Enviando para MinIO bronze/dimensoes/albuns/
  → dimensoes/albuns/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/albuns/_delta_log/.00000000000000000000.json.crc
  → dimensoes/albuns/_delta_log/00000000000000000000.crc
  → dimensoes/albuns/_delta_log/00000000000000000000.json
  → dimensoes/albuns/ingestao_date=2026-06-21/.part-00000-bcbff47e-8e54-44db-ad67-80581e0f4b5a.c000.snappy.parquet.crc
  → dimensoes/albuns/ingestao_date=2026-06-21/part-00000-bcbff47e-8e54-44db-ad67-80581e0f4b5a.c000.snappy.parquet
albuns concluído — 2,000 registros na Bronze.

Tabela: musicas
CSV baixado: /tmp/landing_local/musicas/musicas.csv
Registros lidos: 10,000


Delta escrito localmente em /tmp/bronze/musicas
Enviando para MinIO bronze/dimensoes/musicas/
  → dimensoes/musicas/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/musicas/_delta_log/.00000000000000000000.json.crc
  → dimensoes/musicas/_delta_log/00000000000000000000.crc
  → dimensoes/musicas/_delta_log/00000000000000000000.json
  → dimensoes/musicas/ingestao_date=2026-06-21/.part-00000-47937566-dc6a-4065-a9a8-3305ca032dd5.c000.snappy.parquet.crc
  → dimensoes/musicas/ingestao_date=2026-06-21/part-00000-47937566-dc6a-4065-a9a8-3305ca032dd5.c000.snappy.parquet
musicas concluído — 10,000 registros na Bronze.

Tabela: generos
CSV baixado: /tmp/landing_local/generos/generos.csv
Registros lidos: 30


Delta escrito localmente em /tmp/bronze/generos
Enviando para MinIO bronze/dimensoes/generos/
  → dimensoes/generos/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/generos/_delta_log/.00000000000000000000.json.crc
  → dimensoes/generos/_delta_log/00000000000000000000.crc
  → dimensoes/generos/_delta_log/00000000000000000000.json
  → dimensoes/generos/ingestao_date=2026-06-21/.part-00000-055c31a7-9813-4ef4-9e1f-c9d98cbf1e64.c000.snappy.parquet.crc
  → dimensoes/generos/ingestao_date=2026-06-21/part-00000-055c31a7-9813-4ef4-9e1f-c9d98cbf1e64.c000.snappy.parquet
generos concluído — 30 registros na Bronze.

Tabela: playlists
CSV baixado: /tmp/landing_local/playlists/playlists.csv
Registros lidos: 2,000


Delta escrito localmente em /tmp/bronze/playlists
Enviando para MinIO bronze/dimensoes/playlists/
  → dimensoes/playlists/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/playlists/_delta_log/.00000000000000000000.json.crc
  → dimensoes/playlists/_delta_log/00000000000000000000.crc
  → dimensoes/playlists/_delta_log/00000000000000000000.json
  → dimensoes/playlists/ingestao_date=2026-06-21/.part-00000-f5739712-6983-492b-af5b-0066a548a629.c000.snappy.parquet.crc
  → dimensoes/playlists/ingestao_date=2026-06-21/part-00000-f5739712-6983-492b-af5b-0066a548a629.c000.snappy.parquet
playlists concluído — 2,000 registros na Bronze.

Tabela: dispositivos
CSV baixado: /tmp/landing_local/dispositivos/dispositivos.csv
Registros lidos: 5,000


Delta escrito localmente em /tmp/bronze/dispositivos
Enviando para MinIO bronze/dimensoes/dispositivos/
  → dimensoes/dispositivos/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/dispositivos/_delta_log/.00000000000000000000.json.crc
  → dimensoes/dispositivos/_delta_log/00000000000000000000.crc
  → dimensoes/dispositivos/_delta_log/00000000000000000000.json
  → dimensoes/dispositivos/ingestao_date=2026-06-21/.part-00000-d10f4e90-ac35-42b7-a79e-bfc2a46ac5f3.c000.snappy.parquet.crc
  → dimensoes/dispositivos/ingestao_date=2026-06-21/part-00000-d10f4e90-ac35-42b7-a79e-bfc2a46ac5f3.c000.snappy.parquet
dispositivos concluído — 5,000 registros na Bronze.

Promoção Landing → Bronze concluída para todas as dimensões.


In [8]:
spark.stop()
